# Hyperparameter Tuning: Grid Search vs. Randomized Search

A model has two kinds of knobs:

- **Parameters** the model *learns* from data (e.g. the split thresholds inside a tree).
- **Hyperparameters** *you* set *before* training (e.g. how many trees, how deep). They are **not** learned — they must be searched for.

This notebook searches the hyperparameter space of a **Random Forest** (and an **SVM** pipeline) two ways and compares them head-to-head:

1. **`GridSearchCV`** — try *every* combination in the grid (exhaustive).
2. **`RandomizedSearchCV`** — sample a *fixed budget* of combinations at random.

For each we record the **best cross-validated score**, the **number of model fits**, and the **wall-clock time**. The punchline: random search reaches a *near-equal* score with far fewer fits, so it finishes much faster.

In [ ]:
import time                                            # perf_counter -> honest wall-clock timing
import numpy as np                                     # arrays, RNG, small numeric helpers
import pandas as pd                                    # tidy comparison table at the end
import matplotlib.pyplot as plt                        # the time/score bar chart
import seaborn as sns                                  # nicer default plot styling only
from scipy.stats import randint                        # a DISCRETE distribution to sample n_estimators from

from sklearn.datasets import load_breast_cancer        # small, clean binary-classification dataset
from sklearn.model_selection import (
    train_test_split,                                  # carve off a held-out test set
    GridSearchCV,                                      # exhaustive search over a grid
    RandomizedSearchCV,                                # random-sample search over the same space
    StratifiedKFold,                                   # class-balanced CV folds (shared by both searches)
)
from sklearn.ensemble import RandomForestClassifier    # the main estimator we tune
from sklearn.svm import SVC                             # a second estimator, tuned inside a Pipeline
from sklearn.pipeline import Pipeline                  # chain scaler + estimator so scaling stays leak-free
from sklearn.preprocessing import StandardScaler       # SVMs need standardized features

# One global seed reused everywhere below (splits, forests, samplers) so every run is identical.
SEED = 42
np.random.seed(SEED)                                   # covers any incidental legacy-numpy randomness
sns.set_theme(style="whitegrid")                       # cosmetic: light grid background for the plot
print("setup complete")

## 1. Data and the train/test split

We use `load_breast_cancer`: 569 samples, 30 numeric features, a binary target (malignant vs. benign).

The golden rule of tuning: **the test set is touched exactly once, at the very end.** All hyperparameter searching happens *inside* the training set via cross-validation. If we peeked at the test set while choosing hyperparameters, our reported score would be optimistically biased.

In [ ]:
# Load features X (569 x 30) and target y (0/1) as plain NumPy arrays.
data = load_breast_cancer()
X, y = data.data, data.target
print(f"X shape: {X.shape}   classes: {np.bincount(y)} (0=malignant, 1=benign)")

# Hold out 25% as a FINAL exam. stratify=y keeps the class ratio identical in train and test,
# which matters for an imbalanced-ish medical dataset. random_state pins the exact split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
print(f"train: {X_train.shape[0]} samples   test: {X_test.shape[0]} samples")

## 2. Cross-validation — how we score a hyperparameter combo

For each candidate combination of hyperparameters we need a *reliable* estimate of how good it is — without touching the test set. **K-fold cross-validation** does this:

1. Split the training data into $K$ equal folds.
2. Train on $K-1$ folds, validate on the held-out fold.
3. Rotate so every fold is the validation fold exactly once.
4. Average the $K$ validation scores.

The CV score is that average:

$$\text{CV score} = \frac{1}{K}\sum_{k=1}^{K} \text{score}\big(\text{model trained on folds}\neq k,\ \text{fold } k\big)$$

Crucially, **each candidate costs $K$ fits**, not one. So the total number of model fits is:

$$\text{total fits} = (\text{number of candidate combinations}) \times K$$

That multiplier is exactly why an exhaustive grid gets expensive fast. We use **stratified** folds so each fold preserves the class balance.

In [ ]:
# ONE cv object shared by BOTH searches -> the comparison is apples-to-apples (identical folds).
# StratifiedKFold keeps the malignant/benign ratio steady in every fold.
# shuffle=True + random_state makes the fold assignment reproducible.
K = 5
cv = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
print(f"using {K}-fold stratified cross-validation")
print(f"=> each candidate combination costs {K} model fits")

## 3. Define the hyperparameter space (Random Forest)

Both searches explore the *same* four Random Forest knobs:

| Hyperparameter | What it controls |
|---|---|
| `n_estimators` | number of trees in the forest (more = steadier, slower) |
| `max_depth` | how deep each tree may grow (`None` = unlimited) |
| `min_samples_split` | minimum samples required to split an internal node |
| `max_features` | how many features each split may consider |

The grid below has $3 \times 3 \times 2 \times 2 = 36$ combinations. At 5 folds that is $36 \times 5 = 180$ fits — the exhaustive cost the grid must pay. (We keep the tree counts deliberately small so the whole notebook runs in seconds.)

In [ ]:
# The explicit GRID: GridSearchCV forms the full Cartesian product of these lists.
param_grid = {
    "n_estimators": [30, 60, 90],          # 3 choices (small forests -> fast, still accurate here)
    "max_depth": [None, 5, 10],            # 3 choices (None = grow until pure)
    "min_samples_split": [2, 5],           # 2 choices
    "max_features": ["sqrt", "log2"],      # 2 choices (features considered per split)
}

# Count combinations = product of the list lengths -> confirms the 36 above.
n_combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"grid combinations: {n_combos}")
print(f"grid fits (combos x folds): {n_combos * K}")

# The base estimator. Fixed random_state so every forest is reproducible; n_jobs=1 here means
# each individual forest trains single-threaded, leaving the parallelism to the search itself.
rf = RandomForestClassifier(random_state=SEED, n_jobs=1)

## 4. GridSearchCV — try every combination

`GridSearchCV` fits a model for **every** grid point on **every** fold, then refits the best combo on the full training set (`refit=True`, the default). We wrap the `.fit()` call in `time.perf_counter()` — a high-resolution monotonic clock — to measure honest wall-clock time. Both searches use `n_jobs=-1` (all CPU cores), so the timing comparison between them stays fair.

In [ ]:
grid_search = GridSearchCV(
    estimator=rf,           # the Random Forest to tune
    param_grid=param_grid,  # the full grid from above
    cv=cv,                  # the SHARED 5-fold stratified splitter
    scoring="accuracy",     # optimize/report classification accuracy
    n_jobs=-1,              # spread the candidate x fold fits across all cores (same setting for both searches)
    refit=True,             # after searching, refit best combo on ALL training data
)

# --- timed fit: perf_counter() before and after; the difference is elapsed seconds ---
t0 = time.perf_counter()
grid_search.fit(X_train, y_train)   # this runs all 180 fits
grid_time = time.perf_counter() - t0

# Number of fits actually performed = (#candidates tried) x (#folds).
grid_fits = len(grid_search.cv_results_["params"]) * K

print(f"best CV accuracy : {grid_search.best_score_:.4f}")
print(f"best params      : {grid_search.best_params_}")
print(f"model fits        : {grid_fits}")
print(f"wall-clock time  : {grid_time:.2f} s")

## 5. RandomizedSearchCV — sample a fixed budget

Instead of all 36 combos, we draw a fixed number (`n_iter`) of random candidates. Two ways to specify a hyperparameter:

- a **list** — sampled uniformly (same values as the grid), or
- a **distribution** (e.g. `scipy.stats.randint`) — sampled continuously, so it can hit values the grid never listed.

We deliberately set `n_iter = 10`, giving $10 \times 5 = 50$ fits — well under the grid's 180. With `random_state` fixed, the sampled candidates are reproducible.

In [ ]:
# SAME space as the grid, but n_estimators is now a DISTRIBUTION (randint samples 30..90)
# instead of a fixed list -> random search can explore in-between values like 47 or 83.
param_dist = {
    "n_estimators": randint(30, 91),       # discrete uniform on [30, 90]
    "max_depth": [None, 5, 10],            # still a list -> sampled uniformly
    "min_samples_split": [2, 5],
    "max_features": ["sqrt", "log2"],
}

N_ITER = 10   # budget: only 10 candidates (vs. 36) -> fewer fits, less time

random_search = RandomizedSearchCV(
    estimator=rf,                 # same base Random Forest
    param_distributions=param_dist,
    n_iter=N_ITER,                # how many random candidates to try
    cv=cv,                        # the SAME shared folds as the grid
    scoring="accuracy",
    n_jobs=-1,                    # same parallelism as the grid -> fair time comparison
    refit=True,
    random_state=SEED,            # makes the random sampling reproducible
)

# --- timed fit, exactly like the grid so the two seconds are comparable ---
t0 = time.perf_counter()
random_search.fit(X_train, y_train)   # runs 50 fits
random_time = time.perf_counter() - t0

random_fits = len(random_search.cv_results_["params"]) * K

print(f"best CV accuracy : {random_search.best_score_:.4f}")
print(f"best params      : {random_search.best_params_}")
print(f"model fits        : {random_fits}")
print(f"wall-clock time  : {random_time:.2f} s")

## 6. Bonus: tuning an SVM inside a Pipeline

SVMs are scale-sensitive, so features **must** be standardized. Doing that with a `Pipeline` is the leak-free way: during CV the scaler is fit on *each fold's training part only*, never on the validation fold.

Prefixing a param name with the step name (`svc__C`) tells the search which pipeline step the hyperparameter belongs to. We keep this grid tiny so it stays fast.

In [ ]:
# Pipeline: standardize -> SVM. The scaler is refit per fold, so no test/val leakage.
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),                  # step name 'scaler'
    ("svc", SVC(random_state=SEED)),               # step name 'svc'
])

# 'svc__C' / 'svc__gamma' target the SVC step inside the pipeline (double underscore = 'belongs to').
svm_grid = {
    "svc__C": [0.1, 1, 10],                        # regularization strength (3 choices)
    "svc__gamma": ["scale", 0.01],                 # RBF kernel width (2 choices)
    "svc__kernel": ["rbf"],                         # fix to RBF to keep the grid small
}

svm_search = GridSearchCV(svm_pipe, svm_grid, cv=cv, scoring="accuracy", n_jobs=-1)

t0 = time.perf_counter()
svm_search.fit(X_train, y_train)
svm_time = time.perf_counter() - t0

print(f"SVM best CV accuracy : {svm_search.best_score_:.4f}")
print(f"SVM best params      : {svm_search.best_params_}")
print(f"SVM wall-clock time  : {svm_time:.2f} s")

## 7. Compare the two strategies

Now the head-to-head: best CV score, number of fits, and seconds for grid vs. randomized. Expect randomized to reach a **near-equal** score with **far fewer fits** and **much less time**. We also compute an efficiency ratio (grid time / random time).

In [ ]:
# Assemble a tidy comparison table with pandas.
comparison = pd.DataFrame({
    "strategy":   ["GridSearchCV", "RandomizedSearchCV"],
    "best_cv_score": [grid_search.best_score_, random_search.best_score_],
    "n_fits":     [grid_fits, random_fits],
    "seconds":    [grid_time, random_time],
})
comparison["best_cv_score"] = comparison["best_cv_score"].round(4)
comparison["seconds"] = comparison["seconds"].round(2)

# Derived summary numbers.
speedup = grid_time / random_time                      # how many times faster random was
score_gap = grid_search.best_score_ - random_search.best_score_  # CV accuracy given up (often ~0)

print(comparison.to_string(index=False))
print(f"\nrandom search used {random_fits/grid_fits:.0%} of the grid's fits")
print(f"random search was  {speedup:.1f}x faster")
print(f"CV score given up  {score_gap:+.4f} (near zero = practically no loss)")

## 8. Final exam — evaluate on the held-out test set

We touch the test set now, for the first time. Each search already refit its best estimator on the full training set (`refit=True`), so `.score()` uses that winning model. This is the number that estimates real-world performance.

In [ ]:
# best_estimator_ is the winning model already refit on ALL training data.
grid_test   = grid_search.best_estimator_.score(X_test, y_test)     # accuracy on held-out data
random_test = random_search.best_estimator_.score(X_test, y_test)
svm_test    = svm_search.best_estimator_.score(X_test, y_test)

print(f"Grid RF   test accuracy : {grid_test:.4f}")
print(f"Random RF test accuracy : {random_test:.4f}")
print(f"SVM       test accuracy : {svm_test:.4f}")

## 9. Visualize the trade-off

Two bar charts side by side: **wall-clock time** (where the strategies differ a lot) and **best CV score** (where they barely differ). Seeing them together is the whole story — random search buys a big time saving for a negligible score cost.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
labels = ["Grid", "Random"]
colors = ["#4c72b0", "#dd8452"]   # blue = grid, orange = random

# --- Left: wall-clock time (lower is better) ---
times = [grid_time, random_time]
bars = ax[0].bar(labels, times, color=colors)
ax[0].set_title("Wall-clock time (s) - lower is better")
ax[0].set_ylabel("seconds")
for bar, t in zip(bars, times):                                   # annotate each bar with its value
    ax[0].text(bar.get_x() + bar.get_width() / 2, t, f"{t:.2f}s",
               ha="center", va="bottom")

# --- Right: best CV score (higher is better); zoom the y-axis to expose the tiny gap ---
scores = [grid_search.best_score_, random_search.best_score_]
bars = ax[1].bar(labels, scores, color=colors)
ax[1].set_title("Best CV accuracy - higher is better")
ax[1].set_ylabel("accuracy")
ax[1].set_ylim(min(scores) - 0.02, max(scores) + 0.01)            # zoom so the near-equal bars are readable
for bar, s in zip(bars, scores):
    ax[1].text(bar.get_x() + bar.get_width() / 2, s, f"{s:.4f}",
               ha="center", va="bottom")

plt.tight_layout()
plt.show()

## 10. When to use which

**Cross-validation** gives every candidate a fair, test-set-free score by rotating validation folds — at a cost of $K$ fits per candidate.

**Grid search (exhaustive coverage).** Tries every combination, so it *guarantees* it finds the best point *in the grid*. But its cost is the full product of all option counts times $K$ — it explodes combinatorially. Best when: the space is small, you have few hyperparameters, or you must be certain about a handful of specific values.

**Random search (fixed budget).** You choose how many candidates to try, so cost is decoupled from the number of hyperparameters. It samples distributions, so it can discover values a grid never listed, and empirically it finds strong configurations fast because usually only a couple of hyperparameters truly matter and random sampling covers those dimensions densely. Best when: the space is large/continuous, compute is limited, or you're doing a first broad sweep.

**Practical recipe:** random search first for a broad, cheap sweep, then a small grid search to fine-tune around the promising region. For big budgets, smarter methods (Bayesian optimization, Hyperband, Optuna) go further — but grid and random remain the essential baselines.

The results above make the point: random search reached essentially the same CV accuracy as the exhaustive grid using a fraction of the fits and a fraction of the time.